# Instamart Customer Intelligence & Support Copilot
## Step-by-step project walkthrough

This notebook is a guided way to run the Python scripts in the project folder. It does **not** replace the `.py` files; it calls them in order so you can see each stage separately.

**Before running:** place this notebook inside your project's `notebooks` folder, alongside the existing `src`, `data`, `outputs`, `state`, `generate_data.py`, `mock_llm.py`, and `requirements.txt` items. Run cells from top to bottom.

## Step 1 — Locate the project folder

This cell checks that the notebook can find the project root and the main scripts. If it reports that a file is missing, move the notebook into the project's `notebooks` folder as described above.

In [12]:
from pathlib import Path
import os
import sys
import subprocess

CURRENT = Path.cwd().resolve()
candidates = [CURRENT, *CURRENT.parents]
PROJECT_ROOT = None

for candidate in candidates:
    if (candidate / "generate_data.py").exists() and (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root. Open this notebook from the project's "
        "'notebooks' folder, with generate_data.py and src/ in the parent project folder."
    )

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Found generate_data.py:", (PROJECT_ROOT / "generate_data.py").exists())
print("Found src folder:", (PROJECT_ROOT / "src").is_dir())

Project root: C:\Users\Sinchana M\Downloads\Instamart_Customer_Intelligence_Project (1)\instamart_customer_intelligence
Python executable: c:\Users\Sinchana M\Downloads\Instamart_Customer_Intelligence_Project (1)\instamart_customer_intelligence\.venv\Scripts\python.exe
Found generate_data.py: True
Found src folder: True


## Step 2 — Check required packages

If this cell reports a missing package, run `python -m pip install -r requirements.txt` in the VS Code terminal using the same Python environment selected for this notebook.

In [13]:
required_modules = ["pandas", "numpy", "sklearn"]
missing_modules = []

for module in required_modules:
    try:
        __import__(module)
    except ImportError:
        missing_modules.append(module)

if missing_modules:
    print("Missing packages:", ", ".join(missing_modules))
    print("Install the project's requirements in the VS Code terminal, then restart the notebook kernel.")
else:
    print("Core packages are available.")

Core packages are available.


## Step 3 — Generate the synthetic data

This runs `generate_data.py`. It may overwrite or regenerate files in the project's data folders, so only run it when you want to recreate the demo data.

In [14]:
result = subprocess.run(
    [sys.executable, "generate_data.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Data generation failed with exit code {result.returncode}")

generated: customers=1200 products=40 orders_raw=13084 tickets=164



## Step 4 — Part A: Data cleaning, validation, and SQL analysis

Each script is executed as a separate process from the project root. This helps preserve the same working directory assumptions as running commands in the terminal.

In [15]:
def run_script(relative_path):
    print(f"\n===== Running {relative_path} =====")
    result = subprocess.run(
        [sys.executable, relative_path],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{relative_path} failed with exit code {result.returncode}")
    print(f"Completed: {relative_path}")

run_script("src/part_a.py")


===== Running src/part_a.py =====
Part A: clean rows 1200 40 12309 completed 11110 cancelled 1199 completed revenue 2998668.8
ALL CHECKPOINTS PASSED

Completed: src/part_a.py


In [16]:
run_script("src/validate.py")


===== Running src/validate.py =====
ALL CHECKPOINTS PASSED

Completed: src/validate.py


In [17]:
run_script("src/sql_analysis.py")


===== Running src/sql_analysis.py =====
SQL analysis saved for 7 queries

Completed: src/sql_analysis.py


## Step 5 — Part B: Customer analytics and lapse prediction

This stage generates customer-level features/segments and evaluates the lapse prediction model.

In [18]:
run_script("src/part_b.py")


===== Running src/part_b.py =====
Part B complete: RFM customers 1175 cohort 1156 selected test F1 0.64

Completed: src/part_b.py


## Step 6 — Part C: Ticket classification and privacy handling

This stage runs the ticket classification demo, including the mock LLM, invalid-output handling, and token-budget demonstrations.

In [19]:
run_script("src/part_c.py")


===== Running src/part_c.py =====
Part C complete: {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4} zero-shot accuracy 0.857 few-shot accuracy 1.0 budget exhausted 65

Completed: src/part_c.py


## Step 7 — Part D: Policy retrieval and support agent

This stage runs the retrieval and refund-agent scenarios. A deprecation warning about `Timestamp.utcnow` is not necessarily a failure if the script finishes successfully.

In [20]:
run_script("src/part_d.py")


===== Running src/part_d.py =====
Part D complete: chunks 15 scenarios [('S1', 'REFUND', 5), ('S2', 'REFUND', 5), ('S3', 'ESCALATE', 5), ('S4', 'DENIED', 4), ('S5', 'DENIED', 3), ('S6', 'NOT_FOUND', 2), ('S7', 'FALLBACK_MAX_ITERATIONS', 6)]
Demos {'token_limit_demo': {'status': 'FALLBACK_TOKEN_LIMIT', 'llm_calls': 3}, 'timeout_demo': {'status': 'FALLBACK_TIMEOUT', 'llm_calls': 0}, 'crash_resume': {'status': 'REFUND', 'final_answer': 'REFUND: INR 128.25 issued.', 'tools_executed': ['get_order', 'search_policy', 'calculate_refund', 'issue_refund'], 'llm_calls': 5}}

Completed: src/part_d.py


## Step 8 — Inspect generated outputs

The following cell lists files under `outputs`. You can open CSV files in Excel or VS Code and JSON/JSONL files in a text editor.

In [21]:
outputs_dir = PROJECT_ROOT / "outputs"
if not outputs_dir.exists():
    print("The outputs folder has not been created yet.")
else:
    files = sorted(p for p in outputs_dir.rglob("*") if p.is_file())
    if not files:
        print("No output files found.")
    else:
        for file_path in files:
            print(file_path.relative_to(PROJECT_ROOT))

outputs\agent_runs.jsonl
outputs\agent_safety_demos.json
outputs\city_spend_summary.csv
outputs\customer_segments.csv
outputs\kmeans_metrics.csv
outputs\lapse_metrics.json
outputs\lapse_model.joblib
outputs\model_interpretation.json
outputs\pca_variance.json
outputs\policy_eval_results.csv
outputs\policy_eval_zero_overlap.csv
outputs\policy_retrieval_example.json
outputs\q1_result.csv
outputs\q2_result.csv
outputs\q3_result.csv
outputs\q4_result.csv
outputs\q5_result.csv
outputs\q6_result.csv
outputs\q7_result.csv
outputs\raw_missingness_audit.csv
outputs\refund_ledger.jsonl
outputs\retention_list.csv
outputs\sql_analysis_summary.json
outputs\triage_budget_demo.csv
outputs\triage_metrics.json
outputs\triage_results.csv


## Step 9 — Optional: view a CSV result

This example looks for a customer-segmentation output. If your project uses a different output filename, the cell will print the available CSV files rather than fail.

In [22]:
import pandas as pd

csv_files = sorted((PROJECT_ROOT / "outputs").glob("*.csv"))
preferred = PROJECT_ROOT / "outputs" / "customer_segments.csv"

if preferred.exists():
    display(pd.read_csv(preferred).head())
elif csv_files:
    print("customer_segments.csv was not found. Available CSV files:")
    for path in csv_files:
        print(path.name)
    display(pd.read_csv(csv_files[0]).head())
else:
    print("No CSV output files found yet. Run the relevant project stages first.")

,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,At-Risk Customers,False
1,C0002,28,7,4644.90,1,Premium Champions,True
2,C0003,7,22,3644.85,2,Frequent Essentials,False
3,C0004,0,5,369.00,0,At-Risk Customers,False
4,C0005,147,2,324.00,3,At-Risk Customers,False


## Step 10 — Final notes

- Run the cells from top to bottom for a full demonstration.
- If you already ran `python src/run_all.py`, you can skip the generation and stage cells and use the output-inspection cells.
- This project uses synthetic data and a mock LLM, so no API key is required.
- If a cell fails, read the first error message carefully. Confirm the notebook uses the same `.venv` environment as VS Code's terminal and that the notebook is inside the project's `notebooks` folder.